In [13]:
import pandas as pd
import numpy as np
from scipy import stats
import tkinter as tk
from tkinter import filedialog

# Abro una ventana para seleccionar el archivo.
ventana = tk.Tk()
ventana.withdraw()
ventana.attributes("-topmost", True)

ruta = filedialog.askopenfilename(
    parent=ventana,
    title="Selecciona estudiantes_limpio_R.csv o estudiantes_limpio.xls",
    filetypes=[
        ("Archivos CSV o XLS", "*.csv *.xls"),
        ("Todos los archivos", "*.*")
    ]
)

ventana.destroy()

if not ruta:
    raise ValueError("No seleccionaste un archivo. Ejecuta otra vez la celda.")

# Ambos archivos que enviaste contienen texto CSV.
df = pd.read_csv(ruta)

notas = df["nota"].dropna()

media = notas.mean()
mediana = notas.median()
std_muestral = notas.std(ddof=1)
var_poblacional = np.var(notas.to_numpy(), ddof=0)
q1 = notas.quantile(0.25)
q3 = notas.quantile(0.75)
iqr = q3 - q1

print("Archivo seleccionado:", ruta)
print("Media:", media)
print("Mediana:", mediana)
print("Desviación estándar muestral:", round(std_muestral, 4))
print("Varianza poblacional:", var_poblacional)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)

Archivo seleccionado: C:/Users/51950/OneDrive/Documentos/ME101_ProgramacionEstadistica/estudiantes_limpio_R.csv
Media: 12.666666666666666
Mediana: 12.5
Desviación estándar muestral: 3.9833
Varianza poblacional: 13.222222222222221
Q1: 9.5
Q3: 15.5
IQR: 6.0


In [16]:
print("Asimetría con pandas:", round(notas.skew(), 4))
print("Curtosis con pandas:", round(notas.kurt(), 4))

print("Asimetría con SciPy:", round(stats.skew(notas), 4))
print("Curtosis con SciPy:", round(stats.kurtosis(notas), 4))

Asimetría con pandas: 0.1688
Curtosis con pandas: -1.8055
Asimetría con SciPy: 0.1233
Curtosis con SciPy: -1.4762


In [17]:
def resumen_estadistico(serie, decimales=4):
    datos = serie.dropna()

    media = datos.mean()
    mediana = datos.median()
    desv_std = datos.std(ddof=1)

    if pd.isna(media) or media == 0:
        cv_pct = np.nan
    else:
        cv_pct = desv_std / media * 100

    return {
        "n": int(datos.count()),
        "media": round(float(media), decimales),
        "mediana": round(float(mediana), decimales),
        "desv_std": round(float(desv_std), decimales),
        "cv_pct": round(float(cv_pct), decimales)
    }


print(resumen_estadistico(df["nota"]))

{'n': 6, 'media': 12.6667, 'mediana': 12.5, 'desv_std': 3.9833, 'cv_pct': 31.4471}


In [18]:
def clasificar_dispersion(cv_pct):
    if pd.isna(cv_pct):
        return "No calculable"
    elif cv_pct < 15:
        return "Baja"
    elif cv_pct < 30:
        return "Moderada"
    else:
        return "Alta"


for columna in ["nota", "asistencia_pct"]:
    resumen = resumen_estadistico(df[columna])

    # Clasifico usando el CV sin redondear.
    datos = df[columna].dropna()
    media = datos.mean()

    if pd.isna(media) or media == 0:
        cv = np.nan
    else:
        cv = datos.std(ddof=1) / media * 100

    print("\nColumna:", columna)
    print(resumen)
    print("Nivel de dispersión:", clasificar_dispersion(cv))


Columna: nota
{'n': 6, 'media': 12.6667, 'mediana': 12.5, 'desv_std': 3.9833, 'cv_pct': 31.4471}
Nivel de dispersión: Alta

Columna: asistencia_pct
{'n': 6, 'media': 77.0, 'mediana': 81.0, 'desv_std': 16.3095, 'cv_pct': 21.1812}
Nivel de dispersión: Moderada
